In [73]:
import base64
from shapely import wkb
import re
from google.protobuf.timestamp_pb2 import Timestamp
from coco import MapReport_pb2
import json

with open('../data/example-fmt.json') as f:
  data = json.load(f)

def parse_routes(route_data):
  """Parse route geometries"""
  route_geoms = []
  for route_obj in route_data:
    geom = wkb.loads(base64.b64decode(route_obj['geom_b64']))
    tmp = route_obj.copy()
    del tmp['geom_b64']
    tmp['geom'] = geom
    route_geoms.append(tmp)
  return route_geoms


def camel_to_snake(name: str) -> str:
    s1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
    s2 = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1)
    return s2.lower()

def parse_map_issues(map_issues):
  """Convert map issues into protobufs (?)"""
  output = []
  for mi in map_issues:
    geom = wkb.loads(bytes.fromhex(mi['reported_location_hexwkb']))
    lng=geom.xy[0][0]
    lat=geom.xy[1][0]
    issue_name_attr = 'ISSUE_' + camel_to_snake(data['map_issues'][0]['issue_type']).upper()
    report = MapReport_pb2.MapReport(
        id=mi['id'],
        type=getattr(MapReport_pb2, issue_name_attr),
        notes=mi['notes'],
        reported_location=MapReport_pb2.GeoPoint(lat=lat, lng=lng),
        created_at=Timestamp(seconds=int(mi['created_at'])),
    )
    print(Timestamp(seconds=int(mi['created_at'])).ToSeconds())
    output.append(report)
  return output

# parse_routes(data['routes'])
parse_map_issues(data['map_issues'])

1759605837


[type: ISSUE_LONG_LIGHT
 notes: "hi"
 reported_location {
   lat: 34.0597305
   lng: -118.444672
 }
 id: "11f38913-4a7c-4075-b750-ff9d9fa15554"
 created_at {
   seconds: 1759605837
 }]

In [18]:
fp = '/Users/bradsquicciarini/Downloads/coco-openai_20241206/C10899__1731191615003833265-1731191679999554007.mcap'

In [19]:
from mcap.reader import make_reader

In [22]:
from mcap_protobuf.decoder import DecoderFactory
with open(fp, 'rb') as f:
  reader = make_reader(f, decoder_factories=[DecoderFactory()])
  for s, c, dm, m in reader.iter_decoded_messages(topics=['/route/issue_report']):
    break

In [37]:
from google.protobuf.descriptor_pb2 import FileDescriptorProto
from google.protobuf.text_format import MessageToString

fd = FileDescriptorProto()
fd.ParseFromString(s.data)

fd0 = FileDescriptorProto()
fd0.ParseFromString(fd.name.encode())

print(MessageToString(fd0))

name: "proto/MapReport.proto"
package: "coco"
message_type {
  name: "GeoPoint"
  field {
    name: "lat"
    number: 1
    label: LABEL_OPTIONAL
    type: TYPE_FLOAT
  }
  field {
    name: "lng"
    number: 2
    label: LABEL_OPTIONAL
    type: TYPE_FLOAT
  }
}
message_type {
  name: "MapReport"
  field {
    name: "type"
    number: 1
    label: LABEL_OPTIONAL
    type: TYPE_ENUM
    type_name: ".coco.IssueType"
  }
  field {
    name: "notes"
    number: 2
    label: LABEL_OPTIONAL
    type: TYPE_STRING
  }
  field {
    name: "reported_location"
    number: 3
    label: LABEL_OPTIONAL
    type: TYPE_MESSAGE
    type_name: ".coco.GeoPoint"
  }
}
enum_type {
  name: "IssueType"
  value {
    name: "ISSUE_OTHER"
    number: 0
  }
  value {
    name: "ISSUE_ROUTE_OBSTRUCTION"
    number: 1
  }
}
syntax: "proto3"



In [36]:
fd.name.encode()

b'\n\x15proto/MapReport.proto\x12\x04coco"$\n\x08GeoPoint\x12\x0b\n\x03lat\x18\x01 \x01(\x02\x12\x0b\n\x03lng\x18\x02 \x01(\x02"d\n\tMapReport\x12\x1d\n\x04type\x18\x01 \x01(\x0e2\x0f.coco.IssueType\x12\r\n\x05notes\x18\x02 \x01(\t\x12)\n\x11reported_location\x18\x03 \x01(\x0b2\x0e.coco.GeoPoint*9\n\tIssueType\x12\x0f\n\x0bISSUE_OTHER\x10\x00\x12\x1b\n\x17ISSUE_ROUTE_OBSTRUCTION\x10\x01b\x06proto3'